
Validator.py
-------------
Independent PoA validator nodes. Each wraps a real ECDSA keypair
(blockchain.PoAValidator) and independently evaluates a proposed
operation against LIVE state read from the real Ethereum contracts
(via EthereumClient view calls) — not a parallel Python ledger, so
Ethereum stays the sole source of truth for ownership (rule #6).

No hedge_engine.py was provided, so hedge-purchase validation reuses
the same rule already enforced on-chain in RealEstateVault.sol
(protectedLevel < currentValue). Swap in real pricing logic later by
extending _validate_purchase_hedge.
"""




# The Libraries

In [ ]:
import json
import time
from dataclasses import dataclass

from blockchain import PoAValidator
from ethereum_client import EthereumClient
import config


# The final decision Output Class

In [ ]:
@dataclass
class Decision:
    validator_id: str
    approve: bool
    reason: str
    signature: str  # ECDSA signature over the decision payload

    def to_dict(self):
        return {
            "validator_id": self.validator_id,
            "approve": self.approve,
            "reason": self.reason,
            "signature": self.signature,
        }


# The Validator Node

In [ ]:

class ValidatorNode:
    def __init__(self, validator_id: str, keypair: PoAValidator):
        self.validator_id = validator_id
        self.keypair = keypair  # holds the private key; only this object can sign

    def evaluate(self, operation: dict, eth: EthereumClient,
                 forced_approve: bool = None, forced_reason: str = None) -> Decision:
        """
        Normally runs the real business-rule check against live chain
        state. `forced_approve`/`forced_reason` exist ONLY so a demo can
        script an exact vote pattern (e.g. "Validator 2 votes REJECT")
        for a specific scenario walkthrough, without touching the parts
        that must never be faked: signing, threshold counting, block
        authorization, hash-chain/signature validation. When forced,
        the reason string says so explicitly rather than pretending it
        came from a business-rule check.
        """
        if forced_approve is not None:
            approve = forced_approve
            reason = forced_reason or "scripted for demonstration purposes"
        else:
            approve, reason = self._check_business_rules(operation, eth)

        payload = json.dumps({
            "tx_id": operation["tx_id"],
            "op_type": operation["op_type"],
            "approve": approve,
            "validator_id": self.validator_id,
        }, sort_keys=True).encode()
        signature = self.keypair.sign(payload)
        return Decision(self.validator_id, approve, reason, signature)

    # ------------------------------------------------------------
    def _check_business_rules(self, operation: dict, eth: EthereumClient):
        op_type = operation["op_type"]
        p = operation["params"]

        if op_type == "create_property":
            return self._validate_create_property(p, eth)
        if op_type == "transfer_shares":
            return self._validate_transfer_shares(p, eth)
        if op_type == "purchase_hedge":
            return self._validate_purchase_hedge(p, eth)
        if op_type == "settle_hedge":
            return self._validate_settle_hedge(p, eth)
        return False, f"Unknown operation type '{op_type}'"

    def _validate_create_property(self, p, eth):
        if p["total_shares"] <= 0:
            return False, "totalShares must be > 0"
        if p["initial_value"] <= 0:
            return False, "initialValue must be > 0"
        if not p["name"]:
            return False, "name must not be empty"
        existing = eth.get_property(p["property_id"])
        if existing["active"]:
            return False, f"property {p['property_id']} already exists on-chain"

        # Stricter internal risk policy for this validator only —
        # demonstrates that independent validators can legitimately
        # disagree on a VALID transaction while the threshold still passes.
        if (self.validator_id == config.STRICT_VALIDATOR_ID
                and p["total_shares"] > config.STRICT_MAX_TOTAL_SHARES):
            return False, (f"exceeds this validator's internal risk cap of "
                            f"{config.STRICT_MAX_TOTAL_SHARES} total shares")

        return True, "create_property rules satisfied"

    def _validate_transfer_shares(self, p, eth):
        if p["amount"] <= 0:
            return False, "amount must be > 0"
        if p["from_addr"] == p["to_addr"]:
            return False, "sender and recipient must differ"
        balance = eth.balance_of(p["from_addr"], p["property_id"])
        locked = eth.locked_shares(p["from_addr"], p["property_id"])
        unlocked = balance - locked
        if p["amount"] > unlocked:
            return False, (f"amount {p['amount']} exceeds unlocked balance "
                            f"{unlocked} (balance={balance}, locked={locked})")
        return True, "transfer_shares rules satisfied"

    def _validate_purchase_hedge(self, p, eth):
        prop = eth.get_property(p["property_id"])
        if not prop["active"]:
            return False, "unknown property"
        if p["protected_level"] >= prop["current_value"]:
            return False, "protectedLevel must be below current property value"
        balance = eth.balance_of(p["holder"], p["property_id"])
        if p["share_amount"] > balance:
            return False, "insufficient shares"
        if p["premium"] <= 0:
            return False, "premium must be > 0"
        return True, "purchase_hedge rules satisfied"

    def _validate_settle_hedge(self, p, eth):
        hedge = eth.get_hedge_position(p["hedge_token_id"])
        if hedge["holder"] == "0x0000000000000000000000000000000000000000":
            return False, "unknown hedge position"
        if hedge["settled"]:
            return False, "hedge already settled"
        if time.time() < hedge["expiry"]:
            return False, "hedge has not yet expired"
        return True, "settle_hedge rules satisfied"
